# 教程：让 PyTorch 使用 AMD RX 5700 XT

[English](install_pytorch_mps_rx5700xt.ipynb) | [简体中文](install_pytorch_mps_rx5700xt_CN.ipynb)

## 这篇教程会完成什么（Goal）

这篇教程适用于运行 macOS 的 x86_64 黑苹果，目标是让 PyTorch 通过
Apple 的 **MPS/Metal** 后端使用 AMD Radeon RX 5700 XT。

跟着教程操作后，你将能够：

1. 创建一套不会影响系统 Python 的独立环境；
2. 在 JupyterLab 中运行 PyTorch；
3. 确认计算设备显示为 `mps:0`；
4. 运行一个简单的矩阵测试，观察 RX 5700 XT 的实际性能。

> 这里不需要安装 CUDA 或 ROCm。CUDA 只适用于 NVIDIA 显卡；在 macOS
> 上，PyTorch 调用这张 AMD 显卡使用的是 MPS/Metal。

## 开始前的准备（Setup）

### 已验证的电脑配置

| 项目 | 配置 |
|---|---|
| CPU | AMD Ryzen 9 3900X |
| GPU | AMD Radeon RX 5700 XT 8 GB |
| 系统架构 | x86_64 |
| macOS | 15.2 Sequoia |
| Metal | Metal 3 |
| Python | 3.11 |
| PyTorch | 2.2.2 |
| torchvision / torchaudio | 0.17.2 / 2.2.2 |
| NumPy | 1.26.4 |

### 为什么教程固定使用这些版本？

macOS x86_64 能直接安装的最后一代官方 PyTorch 安装包是 2.2.x，
所以这里固定使用 Python 3.11 和 PyTorch 2.2.2。即使电脑上已经安装了
更新的 Python，也请不要用系统 Python 直接安装。

先下载本仓库，并进入教程目录：

```bash
git clone https://github.com/hermanzhaozzzz/My-Opencore-EFI-for-AMD3900X-5700XT-TUF-x570-Hackintosh.git
cd My-Opencore-EFI-for-AMD3900X-5700XT-TUF-x570-Hackintosh/tutorial_pytorch-via-this_AMD5700xtGPU
```

下面的 `uv` 和 Conda 两种方法任选一种即可，不要在同一个环境里混用。

参考：[PyTorch 2.2 发布说明](https://pytorch.org/blog/pytorch2-2/) ·
[PyTorch MPS 使用说明](https://docs.pytorch.org/docs/stable/notes/mps.html)

## 安装步骤（Steps）

### 方法 A：使用 uv（推荐）

如果还没有安装 `uv`，可以先运行：

```bash
brew install uv
```

然后在刚才进入的教程目录中执行：

```bash
uv python install 3.11
uv venv --python 3.11 .venv

uv pip install --python .venv/bin/python   "numpy==1.26.4"   "torch==2.2.2"   "torchvision==0.17.2"   "torchaudio==2.2.2"   "jupyterlab>=4,<5"   "ipykernel>=6,<8"
```

安装完成后启动 JupyterLab：

```bash
env -u PYTORCH_ENABLE_MPS_FALLBACK .venv/bin/jupyter lab
```

浏览器打开后，选择本教程的 `install_pytorch_mps_rx5700xt.ipynb`。
使用 `.venv/bin/jupyter` 启动，可以避免误用电脑上的其他 Python 环境。

### 方法 B：使用 Conda

如果平时使用 Anaconda 或 Miniconda，可以改用下面的方法：

```bash
conda create -n pytorch-mps-rx5700xt python=3.11 pip -y
conda activate pytorch-mps-rx5700xt

python -m pip install   "numpy==1.26.4"   "torch==2.2.2"   "torchvision==0.17.2"   "torchaudio==2.2.2"   "jupyterlab>=4,<5"   "ipykernel>=6,<8"
```

保持 Conda 环境处于激活状态，然后启动：

```bash
env -u PYTORCH_ENABLE_MPS_FALLBACK jupyter lab
```

这里由 Conda 管理 Python 环境，由 pip 安装经过验证的官方 PyTorch 包。
不要把这个环境直接升级到 PyTorch 2.3 或更高版本。

### 第一步：检查安装是否正确

在 JupyterLab 菜单中点击 **Kernel → Restart Kernel and Run All Cells**。
下面的代码会显示当前 Python、PyTorch 和 MPS 状态。

正确结果应该满足：

- Python 是 `3.11.x`；
- 电脑架构是 `x86_64`；
- PyTorch 是 `2.2.2`；
- `MPS built` 和 `MPS available` 都是 `True`。

In [1]:
import os
import platform
import sys

import numpy as np
import torch

print(f"Python executable : {os.path.basename(sys.executable)}")
print(f"Python version    : {platform.python_version()}")
print(f"Architecture      : {platform.machine()}")
print(f"NumPy             : {np.__version__}")
print(f"PyTorch           : {torch.__version__}")
print(f"MPS built         : {torch.backends.mps.is_built()}")
print(f"MPS available     : {torch.backends.mps.is_available()}")

assert platform.machine() == "x86_64", "这份教程只验证了 x86_64 环境"
assert platform.python_version_tuple()[:2] == ("3", "11"), "请改用 Python 3.11"
assert torch.__version__.startswith("2.2.2"), "请安装 PyTorch 2.2.2"
assert torch.backends.mps.is_built(), "当前 PyTorch 不包含 MPS 支持"
assert torch.backends.mps.is_available(), "系统没有提供可用的 MPS 设备"

fallback = os.environ.get("PYTORCH_ENABLE_MPS_FALLBACK", "").lower()
assert fallback not in {"1", "true", "yes"}, (
    "请关闭 PYTORCH_ENABLE_MPS_FALLBACK，然后重新启动 JupyterLab"
)

print("\n环境检查通过，可以继续。")

Python executable : python
Python version    : 3.11.15
Architecture      : x86_64
NumPy             : 1.26.4
PyTorch           : 2.2.2
MPS built         : True
MPS available     : True

环境检查通过，可以继续。


## 确认显卡正在工作（Checks）

### 第二步：运行最小验证代码

下面这段代码会直接在 MPS 设备上创建张量并完成一次平方运算。
如果最后看到 `device=mps:0` 和 `PASS`，就说明 PyTorch 已经把计算交给显卡。

这台测试电脑的 Ryzen 9 3900X 没有核显，因此这里的 `mps:0` 就是
AMD Radeon RX 5700 XT。

In [2]:
# 可以把这段最小验证代码复制到其他项目中
import torch

assert torch.backends.mps.is_available(), "MPS 不可用"

device = torch.device("mps")
x = torch.tensor([1.0, 2.0, 3.0], device=device)
y = x.square()
torch.mps.synchronize()

assert y.tolist() == [1.0, 4.0, 9.0]
print(f"PASS: {y} · device={y.device}")

PASS: tensor([1., 4., 9.], device='mps:0') · device=mps:0


### 第三步：运行一个简短性能测试

下面会连续进行 10 次 `4096 × 4096` 的 FP16 矩阵乘法，通常只需要几秒。
前两次计算用于预热，不计入成绩。

运行时可以打开 iStat Menus、活动监视器或其他硬件监控工具，观察 GPU
占用和温度。不同的系统负载、显卡频率和散热条件会让结果略有波动。

In [3]:
import time

matrix_size = 4096
matrix_a = torch.randn(
    (matrix_size, matrix_size), device="mps", dtype=torch.float16
)
matrix_b = torch.randn(
    (matrix_size, matrix_size), device="mps", dtype=torch.float16
)

# 先预热两次，避免第一次编译 Metal kernel 的时间影响成绩
torch.mps.synchronize()
for _ in range(2):
    result = matrix_a @ matrix_b
torch.mps.synchronize()

timed_iterations = 10
started = time.perf_counter()
for _ in range(timed_iterations):
    result = matrix_a @ matrix_b
torch.mps.synchronize()
elapsed = time.perf_counter() - started

operations = 2 * matrix_size ** 3 * timed_iterations
tflops = operations / elapsed / 1e12

print(f"Matrix       : {matrix_size} x {matrix_size}")
print(f"Device       : {result.device}")
print(f"Iterations   : {timed_iterations}")
print(f"Elapsed      : {elapsed:.4f} seconds")
print(f"Throughput   : {tflops:.2f} TFLOP/s")
print(f"Checksum     : {result[0, 0].item():.6g}")

assert str(result.device) == "mps:0"
assert tflops > 0

del matrix_a, matrix_b, result
torch.mps.empty_cache()

Matrix       : 4096 x 4096
Device       : mps:0
Iterations   : 10
Elapsed      : 0.2161 seconds
Throughput   : 6.36 TFLOP/s
Checksum     : -22.9531


### 本机实测成绩

以下结果来自本仓库对应的 Ryzen 9 3900X + RX 5700 XT 8 GB 黑苹果。
它们可以作为安装成功后的参考值，不要求每台电脑得到完全相同的数字。

| 测试项目 | 本机结果 |
|---|---:|
| 最小 MPS 验证 | 通过，设备为 `mps:0` |
| Notebook 短测 | **6.36 TFLOP/s** |
| 五分钟压力测试时长 | 300.74 秒 |
| 五分钟内完成的矩阵乘法 | 526 次 |
| 五分钟平均性能 | **6.49 TFLOP/s** |
| 压力测试 GPU 占用 | 约 95%–100% |
| 压力测试最高 GPU Die 温度 | 97°C |
| 测试期间 MPS / Driver 内存 | 约 0.84 / 1.13 GiB |

如何看这些结果：

- 出现 `mps:0` 是最重要的，它证明计算确实在 GPU 上执行；
- 短测达到约 6 TFLOP/s，说明环境和 Metal 加速工作正常；
- 不同 PyTorch 操作的速度差异很大，矩阵测试成绩不等于模型训练速度；
- 97°C 偏高。如果需要连续训练数小时，建议先改善机箱风道、风扇策略或显卡散热。

## 在自己的项目中使用（Next Steps）

### 把模型和数据放到 MPS

在项目中可以使用下面的写法：

```python
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

model = model.to(device)
inputs = inputs.to(device)
outputs = model(inputs)
```

模型参数和输入数据必须位于同一个设备上。如果出现
“input and model are on different devices”一类错误，请分别检查
`next(model.parameters()).device` 和 `inputs.device`。

### 使用时需要知道的几件事

- RX 5700 XT 有 8 GB 显存。如果遇到内存不足，先减小 `batch_size` 或输入尺寸。
- 建议先用 `float32` 跑通模型，再根据模型兼容性尝试 `float16`。
- PyTorch 2.2.2 的 AMD MPS 不支持 `bfloat16`。
- 少数 PyTorch 算子没有 MPS 实现。可以临时用
  `PYTORCH_ENABLE_MPS_FALLBACK=1` 让它们回到 CPU，但速度会降低。
- 测试是否真正使用显卡时，应关闭 fallback，并确认张量设备是 `mps:0`。

到这里，PyTorch 调用 RX 5700 XT 的安装和验证就完成了。